In [2]:
import requests
import pandas as pd
from ipyleaflet import Map, Marker, AwesomeIcon, TileLayer
from ipywidgets import Output

# 🔑 카카오 REST API 키
KAKAO_API_KEY = "4f25ea2e2ce1c7a3a76f4ed96a3a44ff"

def search_places(query, x, y, radius=3000, size=10):
    url = "https://dapi.kakao.com/v2/local/search/keyword.json"
    headers = {"Authorization": f"KakaoAK " + KAKAO_API_KEY}
    params = {"query": query, "x": x, "y": y, "radius": radius, "size": size, "sort": "accuracy"}
    response = requests.get(url, headers=headers, params=params)
    return response.json()

# 지도 생성
m = Map(center=(37.5665, 126.9780), zoom=13, scroll_wheel_zoom=True)
tiles = TileLayer(url="https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",
                  attribution="© OpenStreetMap contributors")
m.add_layer(tiles)

output = Output()
display(m, output)

current_marker = [None]
recommendation_markers = []

def handle_click(**kwargs):
    if kwargs.get("type") == "click":
        lat, lng = kwargs["coordinates"]

        # 내 위치 마커
        if current_marker[0] is not None:
            m.remove_layer(current_marker[0])
        my_icon = AwesomeIcon(name="home", marker_color="red", icon_color="white")
        my_marker = Marker(location=(lat, lng), icon=my_icon, title="내 위치")
        m.add_layer(my_marker)
        current_marker[0] = my_marker

        # 이전 추천 마커 제거
        for mk in recommendation_markers:
            m.remove_layer(mk)
        recommendation_markers.clear()

        output.clear_output()
        with output:
            print(f"\n📍 좌표: 위도 {lat:.6f}, 경도 {lng:.6f}")

        # 검색 실행
        queries = ["헬스장", "공원"]
        results = []
        for q in queries:
            data = search_places(q, lng, lat)
            for place in data.get("documents", []):
                results.append({
                    "이름": place["place_name"],
                    "카테고리": place["category_name"],
                    "주소": place["road_address_name"],
                    "거리(m)": int(place["distance"]) if place["distance"].isdigit() else 999999,
                    "위도": float(place["y"]),
                    "경도": float(place["x"]),
                    "링크": place["place_url"],
                    "검색어": q
                })

        if not results:
            with output:
                print("⚠️ 근처에 헬스장/공원이 없습니다.")
            return

        df = pd.DataFrame(results).sort_values(by="거리(m)").reset_index(drop=True)

        # 디버깅: 결과 확인
        with output:
            print("\n🔍 검색 결과 DataFrame:")
            display(df.head(5))

        # 추천 마커 추가
        for _, row in df.head(10).iterrows():
            if row["검색어"] == "헬스장":
                icon = AwesomeIcon(name="dumbbell", marker_color="blue", icon_color="white")
            else:
                icon = AwesomeIcon(name="tree", marker_color="green", icon_color="white")

            mk = Marker(
                location=(row["위도"], row["경도"]),
                icon=icon,
                title=f"{row['이름']} ({row['거리(m)']}m)"
            )
            m.add_layer(mk)
            recommendation_markers.append(mk)

        with output:
            print("\n✅ 추천 장소 TOP 10 (헬스장=파랑, 공원=초록)")
            display(df[["이름", "카테고리", "주소", "거리(m)", "링크", "검색어"]].head(10))

m.on_interaction(handle_click)


Map(center=[37.5665, 126.978], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zo…

Output()